# ResNet18 V2 Baseline for ICPBL Wafer 3-Class Classification

This notebook trains ResNet18 3-class baselines on the known wafer defect classes (`DIE_BROKEN`, `NORMAL`, `NO_DIE`) using mild contrast/sharpness augmentation variants for v2 experiments. Each configuration saves its best validation-loss checkpoint, config, and train log under `baseline/checkpoints/v2/`.


In [1]:
import os
import random
from collections import Counter
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

import wandb


In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 8
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
NUM_WORKERS = 2
IMAGE_SIZE = 224
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'icpbl-wafer-open-set')
WANDB_MODE = os.getenv('WANDB_MODE', 'online')
CHECKPOINT_ROOT_NAME = 'v2'

AUGMENTATION_CONFIGS = [
    {
        'name': 'aug_cs_mild',
        'rotation_degrees': 5,
        'contrast_strength': 0.08,
        'sharpness_factor': 1.15,
        'sharpness_p': 0.30,
        'autocontrast_p': 0.00,
    },
    {
        'name': 'aug_cs_medium',
        'rotation_degrees': 8,
        'contrast_strength': 0.15,
        'sharpness_factor': 1.35,
        'sharpness_p': 0.50,
        'autocontrast_p': 0.00,
    },
    {
        'name': 'aug_cs_autocontrast',
        'rotation_degrees': 6,
        'contrast_strength': 0.08,
        'sharpness_factor': 1.15,
        'sharpness_p': 0.30,
        'autocontrast_p': 0.50,
    },
]

print('device:', DEVICE)
print('wandb mode:', WANDB_MODE)
print('augmentation configs:', [cfg['name'] for cfg in AUGMENTATION_CONFIGS])


device: cuda


In [3]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'prepared_dataset_G5_622').exists():
            return candidate
    raise FileNotFoundError('Could not find prepared_dataset_G5_622 from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
BASELINE_DIR = REPO_ROOT / 'baseline'
CKPT_DIR = BASELINE_DIR / 'checkpoints' / CHECKPOINT_ROOT_NAME
DATA_ROOT = REPO_ROOT / 'prepared_dataset_G5_622'

KNOWN_LABELS = {
    'DIE_BROKEN': 0,
    'NORMAL': 1,
    'NO_DIE': 2,
}
print('repo root:', REPO_ROOT)
print('data root:', DATA_ROOT)
print('checkpoint root:', CKPT_DIR)


repo root: /home/minje/icpbl-wafer-open-set
data root: /home/minje/icpbl-wafer-open-set/prepared_dataset_G5_622
checkpoint dir: /home/minje/icpbl-wafer-open-set/baseline/checkpoints


In [4]:
def list_image_files(image_dir: Path):
    return sorted([p for p in image_dir.iterdir() if p.is_file()])


def collect_samples(split: str):
    split_root = DATA_ROOT / split / 'G5'
    samples = []
    class_counts = Counter()

    for class_name, label in KNOWN_LABELS.items():
        image_dir = split_root / class_name / 'images'
        if not image_dir.exists():
            continue

        for image_path in list_image_files(image_dir):
            samples.append((image_path, label, class_name))
            class_counts[class_name] += 1

    return samples, class_counts


class WaferDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label, class_name = self.samples[index]
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, label, class_name, str(image_path)


In [5]:
def build_train_transform(aug_cfg):
    transform_ops = [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(aug_cfg['rotation_degrees']),
    ]
    if aug_cfg['contrast_strength'] > 0:
        transform_ops.append(transforms.ColorJitter(contrast=aug_cfg['contrast_strength']))
    if aug_cfg['sharpness_p'] > 0:
        transform_ops.append(
            transforms.RandomAdjustSharpness(
                sharpness_factor=aug_cfg['sharpness_factor'],
                p=aug_cfg['sharpness_p'],
            )
        )
    if aug_cfg['autocontrast_p'] > 0:
        transform_ops.append(transforms.RandomAutocontrast(p=aug_cfg['autocontrast_p']))
    transform_ops.extend(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    return transforms.Compose(transform_ops)


def build_eval_transform():
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def create_dataloaders(aug_cfg):
    train_samples, train_counts = collect_samples('train')
    val_samples, val_counts = collect_samples('val')

    train_dataset = WaferDataset(train_samples, transform=build_train_transform(aug_cfg))
    val_dataset = WaferDataset(val_samples, transform=build_eval_transform())

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    return {
        'train_samples': train_samples,
        'val_samples': val_samples,
        'train_counts': train_counts,
        'val_counts': val_counts,
        'train_dataset': train_dataset,
        'val_dataset': val_dataset,
        'train_loader': train_loader,
        'val_loader': val_loader,
    }


preview_artifacts = create_dataloaders(AUGMENTATION_CONFIGS[0])
print('preview train counts:', preview_artifacts['train_counts'])
print('preview val counts:', preview_artifacts['val_counts'])
print('preview dataset sizes:', len(preview_artifacts['train_dataset']), len(preview_artifacts['val_dataset']))


train counts: Counter({'NORMAL': 60, 'DIE_BROKEN': 26, 'NO_DIE': 23})
val counts: Counter({'NORMAL': 20, 'DIE_BROKEN': 8, 'NO_DIE': 7})
dataset sizes: 109 35


In [6]:
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)


def show_class_samples(dataset, max_per_class: int = 4):
    grouped = {name: [] for name in KNOWN_LABELS}
    for image, _, class_name, _ in dataset:
        if len(grouped[class_name]) < max_per_class:
            grouped[class_name].append(image)
        if all(len(images) >= max_per_class for images in grouped.values()):
            break

    num_classes = len(grouped)
    fig, axes = plt.subplots(num_classes, max_per_class, figsize=(3 * max_per_class, 3 * num_classes))
    for row, (class_name, images) in enumerate(grouped.items()):
        for col in range(max_per_class):
            ax = axes[row, col] if num_classes > 1 else axes[col]
            ax.axis('off')
            if col < len(images):
                ax.imshow(denormalize(images[col]).permute(1, 2, 0))
            if col == 0:
                ax.set_title(class_name)
    plt.tight_layout()
    plt.show()


show_class_samples(preview_artifacts['val_dataset'])


/tmp/ipykernel_2392987/2448829946.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
def build_training_components(train_samples):
    weights = ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, len(KNOWN_LABELS))
    model = model.to(DEVICE)

    train_label_counts = Counter(label for _, label, _ in train_samples)
    class_weights = torch.tensor(
        [
            len(train_samples) / (len(KNOWN_LABELS) * train_label_counts[class_id])
            for class_id in range(len(KNOWN_LABELS))
        ],
        dtype=torch.float32,
        device=DEVICE,
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    return model, criterion, optimizer, scheduler, class_weights


def build_cfg(aug_cfg, train_counts, val_counts, class_weights, output_dir):
    return {
        'experiment': aug_cfg['name'],
        'model': 'resnet18',
        'pretrained_weights': 'ResNet18_Weights.DEFAULT',
        'batch_size': BATCH_SIZE,
        'epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'label_smoothing': LABEL_SMOOTHING,
        'num_workers': NUM_WORKERS,
        'image_size': IMAGE_SIZE,
        'known_labels': KNOWN_LABELS,
        'class_weights': class_weights.detach().cpu().tolist(),
        'train_counts': dict(train_counts),
        'val_counts': dict(val_counts),
        'augmentation': aug_cfg,
        'checkpoint_dir': str(output_dir),
    }


def save_json(payload, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2))


v2_results = []
history_by_config = {}


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/minje/.netrc.


wandb: Currently logged in as: minje227_hyu (minje227_hyu-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run r1bolvx7


wandb: Tracking run with wandb version 0.24.2


wandb: Run data is saved locally in /home/minje/icpbl-wafer-open-set/baseline/wandb/run-20260602_135128-r1bolvx7
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run resnet18_baseline


wandb: ⭐️ View project at https://wandb.ai/minje227_hyu-hanyang-university/icpbl-wafer-open-set


wandb: 🚀 View run at https://wandb.ai/minje227_hyu-hanyang-university/icpbl-wafer-open-set/runs/r1bolvx7


class weights: [1.3974359035491943, 0.605555534362793, 1.5797101259231567]


In [8]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for images, labels, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
        all_targets.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)
    return {
        'loss': avg_loss,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


@torch.no_grad()
def gather_outputs(model, loader):
    model.eval()
    logits_list = []
    labels_list = []

    for images, labels, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)

        logits_list.append(logits.cpu())
        labels_list.append(labels)

    logits = torch.cat(logits_list).numpy()
    labels = torch.cat(labels_list).numpy()
    return logits, labels


def evaluate_classifier(logits, labels, criterion=None):
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)

    metrics = {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }
    if criterion is not None:
        logits_tensor = torch.as_tensor(logits, dtype=torch.float32, device=DEVICE)
        labels_tensor = torch.as_tensor(labels, dtype=torch.long, device=DEVICE)
        metrics['loss'] = criterion(logits_tensor, labels_tensor).item()
    return metrics


In [9]:
def save_checkpoint(state_dict, path, cfg, selection_metric, selection_value, best_epoch):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'model_state_dict': state_dict,
            'cfg': cfg,
            'known_labels': KNOWN_LABELS,
            'image_size': IMAGE_SIZE,
            'selection_metric': selection_metric,
            'selection_value': selection_value,
            'best_epoch': best_epoch,
        },
        path,
    )


for aug_cfg in AUGMENTATION_CONFIGS:
    print(f"===== training {aug_cfg['name']} =====")
    artifacts = create_dataloaders(aug_cfg)
    train_loader = artifacts['train_loader']
    val_loader = artifacts['val_loader']
    train_samples = artifacts['train_samples']
    train_counts = artifacts['train_counts']
    val_counts = artifacts['val_counts']

    model, criterion, optimizer, scheduler, class_weights = build_training_components(train_samples)

    config_dir = CKPT_DIR / aug_cfg['name']
    cfg = build_cfg(aug_cfg, train_counts, val_counts, class_weights, config_dir)
    save_json(cfg, config_dir / 'config.json')

    run = wandb.init(
        project=WANDB_PROJECT,
        name=f"resnet18_{aug_cfg['name']}",
        mode=WANDB_MODE,
        config=cfg,
        reinit=True,
    )

    best_val_loss = float('inf')
    best_loss_state_dict = None
    best_loss_epoch = None
    best_val_f1 = -1.0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer)
        val_logits, val_labels = gather_outputs(model, val_loader)
        val_metrics = evaluate_classifier(val_logits, val_labels, criterion=criterion)
        scheduler.step()

        epoch_metrics = {
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'train_precision': train_metrics['precision'],
            'train_recall': train_metrics['recall'],
            'train_f1': train_metrics['f1'],
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_precision': val_metrics['precision'],
            'val_recall': val_metrics['recall'],
            'val_f1': val_metrics['f1'],
            'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(epoch_metrics)
        wandb.log(epoch_metrics)

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_loss_epoch = epoch
            best_loss_state_dict = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']

        print(
            f"[{aug_cfg['name']}] epoch {epoch:02d} | train_loss={train_metrics['loss']:.4f} "
            f"train_acc={train_metrics['accuracy']:.4f} val_f1={val_metrics['f1']:.4f} "
            f"val_loss={val_metrics['loss']:.4f}"
        )

    best_loss_model_path = config_dir / f"resnet18_best_val_loss_epoch{best_loss_epoch:02d}.pth"
    save_checkpoint(best_loss_state_dict, best_loss_model_path, cfg, 'val_loss', best_val_loss, best_loss_epoch)
    save_json(history, config_dir / 'train_log.json')

    summary = {
        'experiment': aug_cfg['name'],
        'checkpoint_path': str(best_loss_model_path),
        'best_epoch': best_loss_epoch,
        'best_val_loss': best_val_loss,
        'best_val_f1': best_val_f1,
        'history_path': str(config_dir / 'train_log.json'),
        'config_path': str(config_dir / 'config.json'),
    }
    save_json(summary, config_dir / 'summary.json')

    run.summary['best_val_loss'] = best_val_loss
    run.summary['best_val_loss_epoch'] = best_loss_epoch
    run.summary['best_val_f1'] = best_val_f1
    run.finish()

    history_by_config[aug_cfg['name']] = history
    v2_results.append(summary)

print('completed configs:', [item['experiment'] for item in v2_results])


epoch 01 | train_loss=0.5469 train_acc=0.8440 val_f1=0.9696 val_loss=0.3440


epoch 02 | train_loss=0.2831 train_acc=0.9633 val_f1=1.0000 val_loss=0.2268


epoch 03 | train_loss=0.3433 train_acc=0.9450 val_f1=1.0000 val_loss=0.2298


epoch 04 | train_loss=0.3120 train_acc=0.9450 val_f1=0.9454 val_loss=0.3428


epoch 05 | train_loss=0.2832 train_acc=0.9541 val_f1=1.0000 val_loss=0.2670


...

epoch 26 | train_loss=0.2269 train_acc=1.0000 val_f1=1.0000 val_loss=0.2187


epoch 27 | train_loss=0.2232 train_acc=1.0000 val_f1=1.0000 val_loss=0.2178


epoch 28 | train_loss=0.2132 train_acc=1.0000 val_f1=1.0000 val_loss=0.2175


epoch 29 | train_loss=0.2183 train_acc=1.0000 val_f1=1.0000 val_loss=0.2167


epoch 30 | train_loss=0.2217 train_acc=1.0000 val_f1=1.0000 val_loss=0.2158
best val macro f1: 1.0 at epoch 2
best val loss: 0.2155 at epoch 17


In [10]:
for result in v2_results:
    print(
        f"{result['experiment']}: best_epoch={result['best_epoch']}, "
        f"best_val_loss={result['best_val_loss']:.6f}, checkpoint={result['checkpoint_path']}"
    )


validation metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [11]:
for result in v2_results:
    experiment = result['experiment']
    history = history_by_config[experiment]
    output_dir = Path(result['checkpoint_path']).parent

    plt.figure(figsize=(7, 4))
    plt.plot([m['epoch'] for m in history], [m['train_loss'] for m in history], marker='o', label='train_loss')
    plt.plot([m['epoch'] for m in history], [m['val_loss'] for m in history], marker='o', label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'{experiment} loss curves')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / 'loss_curve.png', dpi=150)
    plt.show()


/tmp/ipykernel_2392987/1328950932.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
save_json(v2_results, CKPT_DIR / 'v2_training_summary.json')
print('saved training summary to', CKPT_DIR / 'v2_training_summary.json')


wandb: updating run metadata


saved best val_f1 model to /home/minje/icpbl-wafer-open-set/baseline/checkpoints/resnet18_best_val_f1_epoch02.pth
saved best val_loss model to /home/minje/icpbl-wafer-open-set/baseline/checkpoints/resnet18_best_val_loss_epoch17.pth


wandb: uploading config.yaml; uploading wandb-summary.json


wandb: 
wandb: Run history:
wandb:           epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:              lr █████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
wandb:  train_accuracy ▁▆▆▆▆██▆▇▆▆████▆▇▇████████████
wandb:        train_f1 ▁▆▆▆▆██▆▇▆▆████▆▇▇████████████
wandb:      train_loss █▂▄▃▂▂▂▃▂▃▂▂▂▁▂▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train_precision ▁▆▅▅▅▇█▆▇▆▆███▇▆▇▇███▇██▇█████
wandb:    train_recall ▁▇▆▆▆██▇▇▆▇▇▇██▇██████████████
wandb:    val_accuracy ▅██▁█████▅▅██████████▅████████
wandb:          val_f1 ▄██▁█████▄▄██████████▄████████
wandb:        val_loss █▂▂█▄▄▂▁▁▃▃▃▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁
wandb:              +2 ...
wandb: 
wandb: Run summary:
wandb:         best_val_f1 1
wandb:   best_val_f1_epoch 2
wandb:       best_val_loss 0.21549
wandb: best_val_loss_epoch 17
wandb:               epoch 30
wandb:                  lr 0
wandb:      train_accuracy 1
wandb:            train_f1 1
wandb:          train_loss 0.22168
wandb:     train_precision 1
wandb:                  +6 ...
wandb: 


wandb: 🚀 View run resnet18_baseline at: https://wandb.ai/minje227_hyu-hanyang-university/icpbl-wafer-open-set/runs/r1bolvx7
wandb: ⭐️ View project at: https://wandb.ai/minje227_hyu-hanyang-university/icpbl-wafer-open-set
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260602_135128-r1bolvx7/logs
